In [ ]:
if captions:
    # Display random sample captions
    print("Sample Captions from Dataset:\n")
    sample_indices = np.random.choice(len(captions), size=min(10, len(captions)), replace=False)
    
    for idx, i in enumerate(sample_indices):
        cap_item = captions[i]
        image_id = cap_item.get('image_id', f'Image {i}')
        caption = cap_item.get('caption', 'N/A')
        print(f"{idx+1}. [{image_id}]")
        print(f"   Caption: {caption}\n")
else:
    print("No captions loaded.")

## 6. Sample Captions Display

In [ ]:
from PIL import Image

# List available images
if IMAGES_DIR.exists():
    image_files = list(IMAGES_DIR.glob('*.jpg')) + list(IMAGES_DIR.glob('*.png'))
    print(f"Found {len(image_files)} images in {IMAGES_DIR}")
    
    if image_files:
        # Show sample images
        num_samples = min(6, len(image_files))
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for i in range(num_samples):
            img_path = image_files[i]
            try:
                img = Image.open(img_path)
                axes[i].imshow(img)
                axes[i].set_title(f"Image: {img_path.name}")
                axes[i].axis('off')
                
                # Find corresponding caption
                if captions:
                    for cap in captions:
                        if cap.get('image_id') == img_path.name:
                            axes[i].set_xlabel(cap.get('caption', ''), fontsize=9)
                            break
            except Exception as e:
                axes[i].text(0.5, 0.5, f"Error loading image\n{str(e)}", 
                           ha='center', va='center')
                axes[i].axis('off')
        
        # Hide unused subplots
        for i in range(num_samples, len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.show()
else:
    print(f"⚠ Images directory not found at {IMAGES_DIR}")

## 5. Image Preview

In [ ]:
if captions:
    try:
        from wordcloud import WordCloud
        
        # Create word cloud
        caption_text = ' '.join(caption_texts)
        wordcloud = WordCloud(width=800, height=400, 
                             background_color='white',
                             colormap='viridis',
                             max_words=100).generate(caption_text)
        
        plt.figure(figsize=(14, 6))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Word Cloud of All Captions')
        plt.tight_layout(pad=0)
        plt.show()
        
    except ImportError:
        print("WordCloud library not installed. Install with: pip install wordcloud")
else:
    print("No captions loaded.")

## 4. Word Cloud Visualization

In [ ]:
if captions:
    # Build vocabulary and count word frequencies
    all_words = ' '.join(caption_texts).lower().split()
    word_counts = Counter(all_words)
    
    # Get top 20 most common words
    top_words = dict(word_counts.most_common(20))
    
    # Visualize word frequency
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Bar chart
    words_list = list(top_words.keys())
    counts_list = list(top_words.values())
    ax1.barh(words_list, counts_list, color='steelblue')
    ax1.set_xlabel('Frequency')
    ax1.set_title('Top 20 Most Common Words')
    ax1.invert_yaxis()
    
    # Word count distribution (log scale)
    all_counts = sorted(word_counts.values(), reverse=True)
    ax2.loglog(range(1, len(all_counts)+1), all_counts, marker='o', markersize=3)
    ax2.set_xlabel('Word Rank (log scale)')
    ax2.set_ylabel('Frequency (log scale)')
    ax2.set_title('Word Frequency Distribution (Zipfian)')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Total unique words: {len(word_counts)}")
    print(f"\nTop 10 words: {dict(word_counts.most_common(10))}")
else:
    print("No captions loaded.")

## 3. Word Frequency Analysis

In [ ]:
if captions:
    # Extract captions
    caption_texts = [cap.get('caption', '') for cap in captions if isinstance(cap, dict)]
    
    # Calculate statistics
    caption_lengths = [len(cap.split()) for cap in caption_texts]
    
    stats_df = pd.DataFrame({
        'Metric': ['Total Images', 'Vocabulary Size', 'Avg Caption Length', 
                   'Min Caption Length', 'Max Caption Length', 'Median Caption Length'],
        'Value': [
            len(captions),
            len(set(' '.join(caption_texts).split())),
            f"{np.mean(caption_lengths):.2f}",
            min(caption_lengths),
            max(caption_lengths),
            f"{np.median(caption_lengths):.2f}"
        ]
    })
    
    print("Dataset Statistics:")
    print(stats_df.to_string(index=False))
    
    # Plot caption length distribution
    plt.figure(figsize=(10, 5))
    plt.hist(caption_lengths, bins=30, edgecolor='black', alpha=0.7)
    plt.xlabel('Caption Length (words)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Caption Lengths')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No captions loaded. Please ensure dataset is available.")

## 2. Dataset Statistics

In [ ]:
# Define paths
DATA_DIR = Path('../data')
IMAGES_DIR = DATA_DIR / 'images'
ANNOTATIONS_DIR = DATA_DIR / 'annotations'

# Try to load annotations if they exist
annotation_file = ANNOTATIONS_DIR / 'captions.json'

if annotation_file.exists():
    with open(annotation_file, 'r') as f:
        captions = json.load(f)
    print(f"✓ Loaded {len(captions)} image-caption pairs")
    
    # Display sample
    print("\nSample annotations:")
    for i, item in enumerate(captions[:3]):
        print(f"  {i+1}. {item}")
else:
    print(f"⚠ Annotation file not found at {annotation_file}")
    print("Please download VizWiz dataset and place annotations in data/annotations/")
    captions = []

## 1. Load Dataset

In [ ]:
# Import Required Libraries
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully")

# VisionVoice: Dataset Exploratory Analysis

This notebook provides comprehensive exploration of the VizWiz image captioning dataset, including:
- Dataset statistics and distribution
- Image visualization
- Caption analysis
- Word frequency and cloud visualization
- Vocabulary statistics